<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/aiagent.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Day 1 Lab2</h2>
            <span style="color:#ff7800;">Goal: Create your own AI Twin Agent.<br/>
            This is the second lab.<br/>
            In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader.<br/>
            Just follow the instructions. <br/>
            Before getting started, prepare your env and token!
            </span>
        </td>
    </tr>
</table>

## Welcome to Lab 2

We're going to hand-build an Agent Loop without any Agent Framework.

### First, some preparation

In the folder `twin` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours; by drap & drop, or copying and pasting! You should be able to download it from your LinkedIn profile; go to your profile page and use the menu under your name. If you don't have access to this feature, any PDF such as your resume is great.

I've also made a file called `summary.txt` in `twin` - please read it and update it to reflect you.

## Note: it might ask again for choosing the kernel and the virtual environment (.venv)

In [14]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [15]:
load_dotenv(override=True)
openai = OpenAI()

Having all the libraries imported, it is the time to create the lab variables.

In [16]:
# This section is used to read the linkedin.pdf file and extract the text from each page

reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
# To print the text from the linkedin.pdf file
print(linkedin)

In [18]:
# This section is used to read the summary.txt file and extract the text from it
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
# To print the text from the summary.txt file
print(summary)

## Sidebar1: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...


## Sidebar2: Roles


| Role | Who/what it represents | Main purpose | Priority |
|---|---|---|---|
| **system** | OpenAI/application-level instructions | Define overall behavior | Very high |
| **developer** | The application developer | Define how your application/agent should behave | Very high |
| **user** | The person using your application | Gives the actual task/request | Lower |
| **assistant** | The model itself | Represents previous model responses | Not really an instruction level |

In [20]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Yoda"}
]

In [ ]:
# we're using the gpt-5.4-nano model to create a response to the user's message
# The idea is just to get a greeting response, and we'll use this to test the system prompt
# A cheap model is used to save costs

response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [22]:
# We're changing the style of the assistant to be snarky and witty to see if it can handle it

messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Yoda"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

### To Check Memeory
Something which is not clear for now whether the AI can remember something from the beginning of this lab conversations!<br>
We're aiming to test it in the following cells.

In [24]:
# We're asking the assistant to see if it can remember the user's name and use it in the response:

messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

### Was the result surprising?

In [26]:
# we're defining three different roles here:
# the system, the user, and the assistant
# the assistant is the AI that we're testing
# the user is whose name we're asking for
# the system is the prompt that we're using to test the AI

messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Yoda"},
    {"role": "assistant", "content": "Well hi there, Yoda. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt.

In [28]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person whose website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, or the question is unrelated, use your tool to record the question and who asked it. Pass their name if they gave it in the conversation; otherwise use "Name not provided". Then tell the user that you don't know. Never make up an answer.

IMPORTANT:
If somebody wants to get a quote for a project, ask their contact info: name, email address.
"""


In [ ]:
display(Markdown(system_prompt))

In [30]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
display(Markdown(response.choices[0].message.content))

# And now - TOOLS!

Let's start with a function...

In [32]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("baby@Yoda.com")

In [34]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    print(f"Tool called to record an email: {email} {name} {notes}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(f"{email} | {name} | {notes}\n")
    return "Email received"

In [35]:
def record_unknown_question(question, asked_by="Name not provided"):
    print(f"Tool called to record an unknown question: {question} (asked by: {asked_by})")
    with open("Unknown-Questions.txt", "a", encoding="utf-8") as f:
        f.write(f"{asked_by} | {question}\n")
    return "Question received"

## Step 1 - write some json to describe the tool
The tool descriptions the model sees.


In [36]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [37]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
            "asked_by": {"type": "string", "description": "The name of the person who asked, if they provided it; otherwise 'Name not provided'"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [38]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [39]:
# This gives us a more elegant way that avoids the IF statement.
# This is the function that calls related tools

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

## Step 2 - chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE:<br/>
If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".
Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [40]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    print(messages) # to show how the messages are updated
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)